In [1]:
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import re
import unicodedata
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
from ext_script_decision_match import search_decisions

# HTML-file auswählen
directory = Path(r"original_decisions_10000")


files = list(directory.glob("*.html"))

print(len(files))

results=[]

for file in files:
    # HTML einlesen
    html = file.read_text(
        encoding="utf-8",
        errors="replace"
    )
    
    # HTML parsen
    soup = BeautifulSoup(html, "lxml")


    entscheid = search_decisions(soup )
    
#===================================== offical Label of decision ==========================================
    decision_sign = soup.find("b").get_text(strip=True)

#===================================== First Nummer in Decision-Name ======================================

    filenamen_ID = int(re.search(r"urteil__(\d{1,5})__", file.name).group(1))



#===================================== Find language ======================================================
    urteilsbereich = soup.select_one("#highlight_content")
    text = urteilsbereich.get_text("\n", strip=True)
    sprache = None
    if "Besetzung" in text:
        sprache = "de"

    elif "Composition" in text:
        sprache = "fr"

    elif "Composizione" in text:
        sprache = "it"  



#===================================== Besetzungsblock ======================================================

    
    besetzung_block = None
    treffer = None
    
    if sprache == "de":
        treffer = re.search(
            r"(Bundesrichter(?:in)?\s+.+?,\s*"
            r"(?:Präsident|Präsidentin),?)\s+"
            r"Besetzung\s+(.*?)\s+Verfahrensbeteiligte",
            text,
            re.DOTALL | re.IGNORECASE
        )
    
        if not treffer:
            treffer = re.search(
                r"Besetzung\s+(.*?)\s+Verfahrensbeteiligte",
                text,
                re.DOTALL
            )
        
    elif sprache == "fr":
        treffer = re.search(
            r"Composition\s+(.*?)\s+"
            r"(?:Participant(?:e)?s? à la procédure|"
            r"\d+[A-Z]?_\d+/\d{4})",
            text,
            re.DOTALL
        )
    
    elif sprache == "it":
        treffer = re.search(
            r"Composizione\s+(.*?)\s+Partecipanti al procedimento",
            text,
            re.DOTALL
        )



    praesident = None
    einzelrichter = False
    einzelrichter_name = None
    richtergremium = []
    weitere_richter = []
    
    if treffer:
        if len(treffer.groups()) == 1:
            besetzung_block = treffer.group(1)
        else:
            besetzung_block = treffer.group(1) + " " + treffer.group(2)
    
        besetzung_block = re.sub(r"\s+", " ", besetzung_block).strip()




    


#===================================== Gerichtspräsident ======================================


        
        if sprache == "fr":
            treffer_praesident = re.search(
                r"(?:M\.\s+le\s+Juge\s+fédéral|"
                r"Mme\s+la\s+Juge\s+fédérale|"
                r"MM\.\s+les\s+Juges\s+fédéraux|"
                r"MM\.\s+et\s+Mmes\s+les\s+Juges\s+fédéraux|"
    
                r"MM\s+et\s+Mme\s+les\s+Juges\s+fédéraux|"
                r"Mmes\s+et\s+MM\s+les\s+Juges\s+fédéraux|"
                
                r"Mmes\s+et\s+M\.\s+les\s+Juges\s+fédéraux|"
                r"MM\.\s+et\s+Mme\s+les\s+Juges\s+fédéraux|"
                r"MM\.\s+et\s+Mme\s+et\s+les\s+Juges\s+fédéraux)"
                r"\s+(.+?),\s*"
                r"(?:Président|Présidente|Juge présidant|Juge présidante)",
                besetzung_block,
                re.DOTALL | re.IGNORECASE
            )
        
        elif sprache == "de":
            treffer_praesident = re.search(
                r"Bundesrichter(?:in)?\s+(.+?),\s*"
                r"(?:Präsident|Präsidentin|(?:als\s+)?präsidierendes Mitglied)",
                besetzung_block
            )
        
        elif sprache == "it":
            treffer_praesident = re.search(
                r"Giudic[ei]\s+federal[ei]\s+(.+?),\s*"
                r"(?:Presidente|Giudice presidente)",
                besetzung_block,
                re.IGNORECASE
            )
        
        else:
            treffer_praesident = None
        

        if treffer_praesident:
            praesident = treffer_praesident.group(1).strip()


#===================================== weitere Mitglieder des Spruchkörpers ======================================================


        if sprache == "de":
            richter_text = re.split(
                r"Gerichtsschreiber(?:in)?",
                besetzung_block,
                maxsplit=1
            )[0]
        

            richter_text = re.sub(
                r",\s*Präsident(?:in)?\s+Bundesrichter(?:in)?\s+",
                ", ",
                richter_text,
                flags=re.IGNORECASE
            )


            # Zuerst Trennkomma einsetzen
            richter_text = re.sub(
                r"\s+(?=nebenamtliche[rn]?\s+Bundesrichter)",
                ", ",
                richter_text,
                flags=re.IGNORECASE
            )

 

        
            richter_text = re.sub(
                r"(?:nebenamtliche[rn]?\s+)?Bundes?richter(?:in|innen)?\s+",
                "",
                richter_text,
                flags=re.IGNORECASE
            )


            
            rollen = {
                "Präsident",
                "Präsidentin",
                 "Bundesrichter",
                "Bundesrichterin",
                "präsidierendes Mitglied",
                "als präsidierendes Mitglied",
                "als Einzelrichter",
                "als Einzelrichterin",
                "als Instruktionsrichterin",
                "als Instruktionsrichter", 
                "als präsidierendes Miglied",
                "Einzelrichterin",
                "präsisierendes Mitglied",
                

                
            }


            if decision_sign == "4A_9/2026":
                print("BESETZUNG:")
                print(repr(besetzung_block))
            
                print("\nRICHTER_TEXT:")
                print(repr(richter_text))
            
               




            
        
        
        elif sprache == "fr":
            richter_text = re.split(
                r"Greffi(?:er|ère)",
                besetzung_block,
                maxsplit=1
            )[0]
        
            richter_text = re.sub(
                r"^(?:"
                r"MM\.?\s+et\s+Mme\s+et\s+les\s+Juges\s+fédéraux,?\s*|"
                r"MM\.?\s+et\s+les\s+Juges\s+fédéraux|"
                r"M\.\s+et\s+Mmes?(?:\s+et)?\s+les\s+Juge?s?\s+fédéraux,?\s*|"
                r"M\.\s+et\s+Mmes\s+les\s+Juges\s+fédéraux|"
                r"MM\.?\s+et\s+Mmes\s+les\s+Juges\s+fédéraux|"
                r"Mmes\s+et\s+M\.\s+les\s+Juge?s?\s+fédéra(?:ux|aux),?\s*|"
                r"Mme\s+et\s+(?:M\.\s+)?les\s+Juge?s?\s+fédéraux,?\s*|"
                r"MM\.?\s+et\s+Mme\s+les\s+Juge?s?\s+fédéraux,?\s*|"
                r"MM\.?\s+les\s+Juge?s?\s+fédéraux,?\s*|"
                r"Mmes?\s+et\s+MM\.?\s+les\s+Juge?s?\s+fédéraux,?\s*|"
                r"Mmes?\s+et\s+MM\.\s+les\s+Juges\s+fédéraux|"
                r"Mmes\s+et\s+M\.\s+les\s+Juges\s+fédéraux|"
                r"MM\.\s+et\s+Mme\s+les\s+Juges\s+fédéraux|"
                r"Mmes\s+les\s+Juges\s+fédéral(?:es|aux)|"
                r"MM\.?\s+les\s+Juges\s+fédéraux|"
                r"Mme\s+les?\s+Juges?\s+fédéraux|"
                
                r"Mme\s+la\s+Juge\s+fédérale|"
                r"M\.\s+le\s+Juge\s+fédéral|"
                r"les\s+Juges\s+fédéraux"
                r")\s+",
                "",
                richter_text,
                flags=re.IGNORECASE
            )
        
            # Beispiel: "Herrmann et Josi" → "Herrmann, Josi"
            richter_text = re.sub(
                r",?\s*(?:présidente?|juge\s+présidant|juge\s+suppléante?)\s*,?",
                ", ", richter_text,flags=re.IGNORECASE)

            richter_text = re.sub(r"\s+et\s+", ", ", richter_text)
        
            rollen = {
                "Président",
                "Présidente",
                "Juge présidant",
                "Juge présidante",
                "en qualité de juge unique",
                "en qualité de juge instructrice",
                "en qualité de Juge unique",
                "en qualité de Juge instructeur",
                "Juge unique",
                "en qualité de",
                "MM",
                "Juge instructrice",
    
            }

            
            
        
        
        elif sprache == "it":
            richter_text = re.split(
                r"Cancellier(?:e|a)",
                besetzung_block,
                maxsplit=1
            )[0]
        
            richter_text = re.sub(
                r"Giudic(?:e|i)\s+federal(?:e|i)\s+",
                "",
                richter_text
            )

            richter_text = re.sub(
                r"\s+e\s+",
                ", ",
                richter_text,
                flags=re.IGNORECASE
            )



            

            richter_text = re.sub(
                r",?\s*(?:Giudice\s+Presidente|Giudice\s+supplente)\s*,?",
                ", ",
                richter_text,
                flags=re.IGNORECASE
            )
      
        
            rollen = {
                "Presidente",
                "Giudice presidente",
                "Giudice unico",
                "in qualità di giudice unico",
                "in qualità di giudice unica"
                
            }


        
        
        

     
        
        richtergremium = [
            re.sub(
                r"^(?:"
                r"MM\.?\s+et\s+Mme\s+et\s+les\s+Juges\s+fédéraux|"
                r"Mmes?\s+les\s+Juges\s+fédéraux|"
                r"MM\.?\s+les\s+Juges\s+fédéraux|"
                r"Mme\s+les?\s+Juges?\s+fédéraux|"
                r"Mme\.?\s+les?\s+Juges?\s+fédérales?|"
                r"Mme?\.?\s+la\s+Juge\s+fédérale|"
                r"(?:M\.\s+la\s+Juge|MM\.?\s+et\s+Mme\s+les\s+Juges)\s+fédérales?|"
                r"M\.\s+le\s+Juge(?:\s+fédéral)?|"
                r"nebenamtlicher\s+Bundesrichter|"
                r"Bundesrichter(?:in)?|"
                r"Bundesricher(?:in)?|"
                r"Bundesricherin|"          
                r"Bundesrichtger|"
                r"Bundesricherin|" 
                r"Bunesrichter|" 
                r"Bundesricherin|"         
                r"Bundsrichter(?:in)?|"
                r"Juge\s+instruct(?:eur|rice)|"
                r"Mme\.?"
                r")\s+",
                "",
                teil.strip(" ."),
                flags=re.IGNORECASE
            )
            for teil in richter_text.split(",")
            if teil.strip(" .")
            and teil.strip(" .") not in rollen
        ]

        richtergremium = [
            re.sub(
                r"^(?:als\s+)?präsidierendes\s+Mitglied\s+|"
                r"\s+als\s+präsidierendes\s+Mitglied$",
                "",
                name,
                flags=re.IGNORECASE
            ).strip()
            for name in richtergremium
        ]





        


        richtergremium = [
            unicodedata.normalize("NFC", name)
            .replace("\u00a0", " ")
            .replace("\u200b", "")
            .strip(" .;,")
            for name in richtergremium
        ]

        richtergremium = [
            name
            for name in richtergremium
            if name
        ]


        

        korrekturen = {
        "Petrik": "Petrik-Haltiner", 
        "Hermann": "Herrmann",
        "Muschetti": "Muschietti",
        "van de Graaaf": "van de Graaf",
        "Van de Graaf": "van de Graaf",
        "Hoffmann": "Hofmann",
        "H ofmann": "Hofmann",
        "Von Felten": "von Felten",
        "de Rossa": "De Rossa",
        "Koc h": "Koch",
         "Kradofler": "Kradolfer",
        "Pont Venthey": "Pont Veuthey",
        "Kneuhühler": "Kneubühler",
        "Müller Th": "Müller",
        "et Hofmann": "Hofmann",
         "Präsident van de Graaf": "van de Graaf",
        "Präsident Abrecht": "Abrecht",
        "les Juges fédéraux Abrecht": "Abrecht",
        "MM. Juges fédéraux Abrecht": "Abrecht",
        "M. Mmes les Juges fédéraux Abrecht": "Abrecht",
        "M. les Juge fédéral Merz": "Merz",
        "M. Donzallaz": "Donzallaz",

      
       
        
            # nur falls wir nach Prüfung sehen, dass das immer stimmt
        }

        richtergremium = [
            korrekturen.get(name, name) 
            for name in richtergremium
        ]

        praesident = korrekturen.get(praesident, praesident)

        
        
        weitere_richter = [
            name
            for name in richtergremium
            if name != praesident
        ]
        
        if len(richtergremium) == 1:
            einzelrichter = True
            einzelrichter_name = richtergremium[0]


#===================================== einzelrichter ======================================================

  

        if sprache == "de":
            treffer_einzelrichter = re.search(
                r"Bundesrichter(?:in)?\s+(.+?),\s+als Einzelrichter(?:in)?",
                besetzung_block
            )
        
        elif sprache == "fr":
            treffer_einzelrichter = re.search(
                r"(?:M\.|Mme)\s+le Juge fédéral\s+(.+?),\s+"
                r"en qualité de juge unique",
                besetzung_block,
                re.IGNORECASE
            )
        
        elif sprache == "it":
            treffer_einzelrichter = re.search(
                r"Giudice federale\s+(.+?),\s+"
                r"in qualità di giudice unic[oa]",
                besetzung_block,
                re.IGNORECASE
            )
        
        else:
            treffer_einzelrichter = None
        
        if treffer_einzelrichter:
            einzelrichter = True
            einzelrichter_name = treffer_einzelrichter.group(1).strip()

#===================================== legal_area  ======================================================$
        match = re.search(r"\d([A-Z])_", decision_sign)

        if match:
            buchstabe = match.group(1)
        
            if buchstabe == "A":
                legal_area = "Zivilrecht"
        
            elif buchstabe == "B":
                legal_area = "Strafrecht"
        
            elif buchstabe == "C":
                legal_area = "Öffentliches Recht"
        
            elif buchstabe == "D":
                legal_area = "Subsidiäre Verfassungsbeschwerde"
        
            else:
                legal_area = "AndererBereich"
        
        else:
            legal_area = "Mistake_Legalara"

        

 
#===================================== Make dictionary ======================================================
    resultat = {
        "id_filename": filenamen_ID,
        "filename": file.name,
        "decision_sign": decision_sign,
        "sprache": sprache,
        "besetzung_block": besetzung_block,
        "praesident": praesident,
        "einzelrichter": einzelrichter,
        "einzelrichter_name": einzelrichter_name,
        "weiterer_richter1": weitere_richter[0] if len(weitere_richter) > 0 else None,
        "weiterer_richter2": weitere_richter[1] if len(weitere_richter) > 1 else None,
         "weiterer_richter3": weitere_richter[2] if len(weitere_richter) > 2 else None,
        "weiterer_richter4": weitere_richter[3] if len(weitere_richter) > 3 else None,
        "entscheid": entscheid,
        "legal area":legal_area,
        
        
        }

    results.append(resultat)
    
    
# DataFrame erstellen
df = pd.DataFrame(results)  



10000
die präsidentin:
1.
die eingabe vom 26. november 2025 wird als gesuch um fristwiederherstellung entgegengenommen und samt beilagen zur weiteren behandlung an das obergericht des kantons zürich überwiesen.
2.
für das bundesgerichtliche verfahren werden keine kosten erhoben.
3.
dieses urteil wird den parteien und dem obergericht des kantons zürich, ii. strafkammer, schriftlich mitgeteilt.
lausanne, 10. dezember 2025
im namen der i. strafrechtlichen abteilung
des schweizerischen bundesgerichts
die präsidentin:    jacquemoud-rossari
die gerichtsschreiberin:    arquint hill
navigation
neue suche
ähnliche leitentscheide suchen
ähnliche urteile ab 2000 suchen
drucken
nach oben
zurück
false
 
.................................................................................................
, il tribunale federale pronuncia:
1.
il ricorso è inammissibile.
2.
le spese giudiziarie di fr. 3'000.-- sono poste a carico dei ricorrenti, in solido.
3.
comunicazione al patrocinatore dei ricorrenti

In [2]:
df[["id_filename","praesident","weiterer_richter1","weiterer_richter2","besetzung_block"]].head(3).fillna("")

,id_filename,praesident,weiterer_richter1,weiterer_richter2,besetzung_block
0,10000,Moser-Szeless,Stadelmann,Parrino,"Mmes et MM. les Juges fédéraux Moser-Szeless, Présidente, Stadelmann, Parrino, Beusch et Bollinger. Greffier : M. Feller."
1,1000,Bovey,,,"Bundesrichter Bovey, Präsident, Gerichtsschreiber Zingg."
2,1001,Aubry Girardin,Donzallaz,Hänni,"Bundesrichterin Aubry Girardin, Präsidentin, Bundesrichter Donzallaz, Bundesrichterin Hänni, Bundesrichterin Ryter, Bundesrichter Kradolfer, Gerichtsschreiber Kaufmann."


## Initial Data Checks
Here, the entscheid and legal_area columns are checked.

In [3]:
#Test 1: The sum should match the number of decisions imported.
#Test 2: No NaN values should be returned.
df.groupby("entscheid", dropna=False).size()

entscheid
abgeschrieben              385
abgewiesen                3840
andere                      15
berichtigt                   3
gutgeheissen               860
nicht eingetreten         4456
teilweise gutgeheissen     441
dtype: int64

In [4]:
#Test 1: Die summe sollte der Anzahl der eingelesenen entscheide entsprechen
#Test 2: Es sollte keine Nan ausgegeben werden.
df.groupby("legal area", dropna=False).size()

legal area
AndererBereich                       435
Strafrecht                          2961
Subsidiäre Verfassungsbeschwerde     480
Zivilrecht                          2345
Öffentliches Recht                  3779
dtype: int64

## One row for each judge
In the next cell, the table is reshaped so that each judge has a separate row.

In [5]:
df_decision_judge = df.melt(
    id_vars=[
        "id_filename",
        "decision_sign",
        "entscheid",
        "legal area"
    ],
    value_vars=[
        "praesident",
        "weiterer_richter1",
        "weiterer_richter2",
        "weiterer_richter3",
        "weiterer_richter4",
    ],
    var_name="funktion",
    value_name="name"
)


In [6]:
df_decision_judge.head(3)

,id_filename,decision_sign,entscheid,legal area,funktion,name
0,10000,9C_75/2024,gutgeheissen,Öffentliches Recht,praesident,Moser-Szeless
1,1000,5A_449/2026,nicht eingetreten,Zivilrecht,praesident,Bovey
2,1001,2E_8/2024,abgewiesen,AndererBereich,praesident,Aubry Girardin


## Find and add judges' political party

Here, I add the table I compiled showing which political party each judge belongs to.

In [7]:
df_richter = pd.read_excel("bundesrichter_ab_2007.xlsx")

In [8]:
df_richter.head(3)

,both_names,election,retired,birth,died,kanton(district),party_raw,name,party
0,Arthur Brunner,2026.0,NaN,NaN,NaN,NaN,SVP,Brunner,SVP
1,Bernard Abrecht,2019.0,NaN,1966.0,NaN,Waadt / Bern,SP,Abrecht,SP
2,Florence Aubry Girardin,2007.0,NaN,1964.0,NaN,Jura,Grüne,Aubry Girardin,Grüne


## Merge the Decisions Table with the Judges Table

The new table shows not only the judges’ names and other information for each decision, but also their political party affiliation.

In [9]:
df_merged = df_decision_judge.merge(
    df_richter,
    on="name",
    how="left"
)

## Cleaning and Formatting the New Table

Here, the table is cleaned and reduced to the necessary columns.

In [10]:
df_merged = df_merged.drop(columns="party_raw")

In [11]:
df_merged[["id_filename","decision_sign","name", "party", "legal area", "entscheid"]].head(2)

,id_filename,decision_sign,name,party,legal area,entscheid
0,10000,9C_75/2024,Moser-Szeless,SVP,Öffentliches Recht,gutgeheissen
1,1000,5A_449/2026,Bovey,FDP,Zivilrecht,nicht eingetreten


## Removing Empty Judge Fields

The table currently still contains many rows with an empty judge field. This is not an error. It is because the system initially assumes that five judges are involved in each decision, although many decisions are made by only three judges or by a single judge. These empty rows are removed here.

In [12]:
# Gewünschte Spalten auswählen
df_cleaned = (
    df_merged[
        ["id_filename", "decision_sign", "name", "party", "legal area", "entscheid"]
    ]
    .dropna(subset=["name"])
)

## Final testing Whether the Data Is Correct

Several tests are performed below to check whether the data is plausible and consistent.

In [13]:
#Test 1: Does every last name in the merged DataFrame have an exact match in the judges DataFrame?
# Number without Treffer should be zero.
k = 0

for richter in df_cleaned["name"]:

    if pd.isna(richter):
        continue

    n = 0

    for prof in df_decision_judge["name"]:
        if prof == richter:
            #print(richter + " VS " +prof)
            n = 1
            break

    if n == 0:
        print("Kein Treffer:", richter)
        k = k + 1

print("Anzahl ohne Treffer:", k)

Anzahl ohne Treffer: 0


In [14]:
#Test 2: Count number of judges for every decision
#There should be zero decision with 2 or 4 judges, because ther are no judges Teams with 2 oder 4 judges
# Test 3: The sum should should be equal to the number of the decisions which are read in in the beginning

Number_with_1 = 0
Number_with_2 = 0
Number_with_3 = 0
Number_with_4 = 0
Number_with_5 = 0

problem_ids = []

for id_filename, urteil in df_merged[df_merged["name"].notna()].groupby("id_filename").size().items():
    if urteil == 1:
        Number_with_1 += 1
    elif urteil == 2:
        Number_with_2 += 1
    elif urteil == 3:
        Number_with_3 += 1
    elif urteil == 4:
        problem_ids.append(id_filename)
        Number_with_4 += 1
    elif urteil == 5:
        Number_with_5 += 1

print(problem_ids)

print("Number of decisions with 1 judge: " + str(Number_with_1))
print("Number of decisions with 2 judges: " + str(Number_with_2))
print("Number of decisions with 3 judges: " + str(Number_with_3))
print("Number of decisions with 4 judges: " + str(Number_with_4))
print("Number of decisions with 5 judges: " + str(Number_with_5))

[]
Number of decisions with 1 judge: 3861
Number of decisions with 2 judges: 0
Number of decisions with 3 judges: 5504
Number of decisions with 4 judges: 0
Number of decisions with 5 judges: 626


## Saving the Final Table as a File

The DataFrame is saved as both a CSV file and an Excel file for manual review.

In [15]:
df_cleaned = df_cleaned[df_cleaned["name"].str.strip() != ""]

# Als CSV speichern
df_cleaned.to_csv("entscheidungen.csv", index=False, encoding="utf-8-sig")

# Als Excel speichern
df_cleaned.to_excel("entscheidungen.xlsx", index=False)